# 08 — Streamlit Data Preparation

Prepare compact, public-facing datasets for the **Mortgage Risk Under Changing Economic Conditions** Streamlit app.

### Goals
- Reuse the finalized 2015–2022 mortgage and economic-feature tables.
- Refit the locked **origination-only logistic regression** on 2015–2019.
- Generate out-of-time predictions for 2020–2022.
- Create aggregated tables for vintage trends, model performance, risk deciles, borrower/loan segments, geography, and economic conditions.
- Export only dashboard-ready aggregates to `data/app/`.

### Responsible-use framing
This notebook supports portfolio-level historical analysis. It does **not** create an individual underwriting or lending-decision tool. Economic variables are used to describe observed environments and associations; they are not interpreted as causal effects on individual borrowers.


## 1. Setup

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
APP_DATA_DIR = PROJECT_ROOT / "data" / "app"
APP_DATA_DIR.mkdir(parents=True, exist_ok=True)

MORTGAGE_PATH = PROCESSED_DIR / "model_data_2015_2022.parquet"
ECONOMIC_PATH = PROCESSED_DIR / "economic_features_2015_2022.parquet"

print("Project root:", PROJECT_ROOT)
print("Mortgage data exists:", MORTGAGE_PATH.exists())
print("Economic data exists:", ECONOMIC_PATH.exists())
print("App output directory:", APP_DATA_DIR)


## 2. Load finalized project data

In [ ]:
mortgages = pd.read_parquet(MORTGAGE_PATH)
economic = pd.read_parquet(ECONOMIC_PATH)

print("Mortgage shape:", mortgages.shape)
print("Economic shape:", economic.shape)
print("Mortgage vintages:", sorted(mortgages["origination_vintage"].unique()))
print("Unique mortgage loans:", mortgages["loan_id"].nunique())
print("Unique economic-feature loans:", economic["loan_id"].nunique())


The economic table contains **post-origination exposure variables**. These are useful for describing the economic environment a mortgage experienced, but they are intentionally **not fed into the origination-time predictive model** below. That avoids using future information to predict a risk score that is supposed to exist at origination.

In [ ]:
dashboard = mortgages.merge(
    economic,
    on="loan_id",
    how="left",
    validate="one_to_one"
)

print("Merged shape:", dashboard.shape)
print("Missing economic rows:", dashboard["starting_market_rate"].isna().sum())


## 3. Recreate the locked origination-only model

In [ ]:
numeric_features = [
    "credit_score",
    "dti",
    "cltv",
    "original_upb",
    "ltv",
    "original_interest_rate",
    "original_loan_term",
    "mi_percent",
    "num_units",
    "num_borrowers",
]

categorical_features = [
    "first_time_homebuyer",
    "occupancy_status",
    "channel",
    "property_type",
    "loan_purpose",
    "super_conforming_flag",
]

target = "seriously_delinquent_24m"
model_features = numeric_features + categorical_features

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

train = dashboard[dashboard["origination_vintage"] <= 2019].copy()
score = dashboard[dashboard["origination_vintage"] >= 2020].copy()

model.fit(train[model_features], train[target].astype(int))
score["predicted_probability"] = model.predict_proba(score[model_features])[:, 1]

print(f"Training loans (2015–2019): {len(train):,}")
print(f"Scored loans (2020–2022): {len(score):,}")


## 4. Verify out-of-time performance

In [ ]:
performance_rows = []

for vintage in [2020, 2021, 2022]:
    d = score[score["origination_vintage"] == vintage]
    y = d[target].astype(int)
    p = d["predicted_probability"]

    performance_rows.append({
        "vintage": vintage,
        "loans": len(d),
        "serious_delinquencies": int(y.sum()),
        "base_rate": y.mean(),
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p),
    })

performance = pd.DataFrame(performance_rows)
performance


In [ ]:
# Expected values from the finalized analysis, with small tolerance for reproducibility.
expected_roc = {2020: 0.7702, 2021: 0.7830, 2022: 0.6981}

for vintage, expected in expected_roc.items():
    actual = performance.loc[
        performance["vintage"] == vintage, "roc_auc"
    ].iloc[0]
    assert abs(actual - expected) < 0.002, (
        f"{vintage} ROC-AUC changed unexpectedly: {actual:.4f} vs {expected:.4f}"
    )

print("Out-of-time performance matches the finalized analysis.")


## 5. Vintage-level mortgage and economic context

In [ ]:
vintage_summary = (
    dashboard.groupby("origination_vintage", as_index=False)
    .agg(
        loans=(target, "size"),
        serious_delinquencies=(target, "sum"),
        delinquency_rate=(target, "mean"),
        median_credit_score=("credit_score", "median"),
        median_dti=("dti", "median"),
        median_ltv=("ltv", "median"),
        median_origination_rate=("original_interest_rate", "median"),
        median_starting_unemployment=("starting_unemployment", "median"),
        median_unemployment_increase=("unemployment_increase", "median"),
        median_hpi_change=("hpi_change_from_start", "median"),
        median_worst_hpi_drawdown=("worst_hpi_drawdown", "median"),
        median_starting_market_rate=("starting_market_rate", "median"),
        median_market_rate_change=("market_rate_change", "median"),
    )
)

vintage_summary


These economic summaries are descriptive. For example, a vintage with both a larger unemployment increase and a higher delinquency rate does not by itself establish that the unemployment change caused the delinquency change.

## 6. Predicted-risk deciles for 2020–2022

In [ ]:
decile_tables = []

for vintage in [2020, 2021, 2022]:
    d = score[score["origination_vintage"] == vintage].copy()

    d["risk_decile"] = pd.qcut(
        d["predicted_probability"],
        q=10,
        labels=False,
        duplicates="drop",
    ) + 1

    base_rate = d[target].mean()

    summary = (
        d.groupby("risk_decile", as_index=False)
        .agg(
            loans=(target, "size"),
            serious_delinquencies=(target, "sum"),
            actual_delinquency_rate=(target, "mean"),
            avg_predicted_risk=("predicted_probability", "mean"),
        )
    )

    summary["lift"] = summary["actual_delinquency_rate"] / base_rate
    summary["vintage"] = vintage
    decile_tables.append(summary)

risk_deciles = pd.concat(decile_tables, ignore_index=True)
risk_deciles = risk_deciles[
    [
        "vintage",
        "risk_decile",
        "loans",
        "serious_delinquencies",
        "avg_predicted_risk",
        "actual_delinquency_rate",
        "lift",
    ]
]

risk_deciles


## 7. Loan-characteristic segments

In [ ]:
segment_data = dashboard.copy()

segment_data["fico_group"] = pd.cut(
    segment_data["credit_score"],
    bins=[-np.inf, 649, 699, 749, 799, np.inf],
    labels=["<650", "650–699", "700–749", "750–799", "800+"],
)

segment_data["dti_group"] = pd.cut(
    segment_data["dti"],
    bins=[-np.inf, 20, 30, 40, 50, np.inf],
    labels=["≤20", "21–30", "31–40", "41–50", "50+"],
)

segment_data["ltv_group"] = pd.cut(
    segment_data["ltv"],
    bins=[-np.inf, 60, 80, 90, 95, np.inf],
    labels=["≤60", "61–80", "81–90", "91–95", "95+"],
)

def summarize_segment(df, column, label):
    out = (
        df.dropna(subset=[column])
        .groupby(["origination_vintage", column], observed=True, as_index=False)
        .agg(
            loans=(target, "size"),
            serious_delinquencies=(target, "sum"),
            delinquency_rate=(target, "mean"),
        )
    )
    out = out.rename(columns={column: "segment"})
    out["segment_type"] = label
    out["segment"] = out["segment"].astype(str)
    return out

segment_summary = pd.concat([
    summarize_segment(segment_data, "fico_group", "Credit score"),
    summarize_segment(segment_data, "dti_group", "Debt-to-income ratio"),
    summarize_segment(segment_data, "ltv_group", "Loan-to-value ratio"),
], ignore_index=True)

segment_summary.head(15)


## 8. Economic-condition groups

In [ ]:
economic_group_data = dashboard.dropna(
    subset=[
        "unemployment_increase",
        "hpi_change_from_start",
        "market_rate_change",
    ]
).copy()

def quartile_labels(series, low_label, high_label):
    # Rank first so qcut remains stable if many observations share the same value.
    return pd.qcut(
        series.rank(method="first"),
        q=4,
        labels=[
            f"Q1 — {low_label}",
            "Q2",
            "Q3",
            f"Q4 — {high_label}",
        ],
    )

economic_group_data["unemployment_change_group"] = quartile_labels(
    economic_group_data["unemployment_increase"],
    "smallest increase",
    "largest increase",
)

economic_group_data["hpi_change_group"] = quartile_labels(
    economic_group_data["hpi_change_from_start"],
    "weakest home-price change",
    "strongest home-price change",
)

economic_group_data["rate_change_group"] = quartile_labels(
    economic_group_data["market_rate_change"],
    "smallest rate change",
    "largest rate change",
)

def summarize_economic_group(df, column, label, value_column):
    out = (
        df.groupby(["origination_vintage", column], observed=True, as_index=False)
        .agg(
            loans=(target, "size"),
            serious_delinquencies=(target, "sum"),
            delinquency_rate=(target, "mean"),
            median_economic_value=(value_column, "median"),
        )
    )
    out = out.rename(columns={column: "condition_group"})
    out["condition_type"] = label
    out["condition_group"] = out["condition_group"].astype(str)
    return out

economic_groups = pd.concat([
    summarize_economic_group(
        economic_group_data,
        "unemployment_change_group",
        "Unemployment change",
        "unemployment_increase",
    ),
    summarize_economic_group(
        economic_group_data,
        "hpi_change_group",
        "Home-price change",
        "hpi_change_from_start",
    ),
    summarize_economic_group(
        economic_group_data,
        "rate_change_group",
        "Market mortgage-rate change",
        "market_rate_change",
    ),
], ignore_index=True)

economic_groups.head(15)


### Interpretation guardrail

The table above compares **observed delinquency rates across groups of realized economic exposure**. It should be presented as historical association, not as an estimate of the causal effect of changing unemployment, home prices, or interest rates. In particular, some exposure measures are observed after origination and can be truncated by the timing of a delinquency event.


## 9. State-level dashboard table

In [ ]:
state_summary = (
    dashboard.groupby(
        ["origination_vintage", "property_state"],
        as_index=False
    )
    .agg(
        loans=(target, "size"),
        serious_delinquencies=(target, "sum"),
        delinquency_rate=(target, "mean"),
        median_credit_score=("credit_score", "median"),
        median_dti=("dti", "median"),
        median_ltv=("ltv", "median"),
        median_starting_unemployment=("starting_unemployment", "median"),
        median_unemployment_increase=("unemployment_increase", "median"),
        median_hpi_change=("hpi_change_from_start", "median"),
        median_market_rate_change=("market_rate_change", "median"),
    )
)

# Suppress very small cells in the public dashboard.
MIN_STATE_CELL = 50
state_summary = state_summary[state_summary["loans"] >= MIN_STATE_CELL].copy()

print("State-vintage rows:", len(state_summary))
state_summary.head()


## 10. Model coefficients for interpretation

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

model_coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
})

model_coefficients["odds_ratio"] = np.exp(model_coefficients["coefficient"])
model_coefficients["absolute_coefficient"] = model_coefficients["coefficient"].abs()
model_coefficients = model_coefficients.sort_values(
    "absolute_coefficient",
    ascending=False,
).reset_index(drop=True)

model_coefficients.head(20)


Numeric features were standardized before fitting, so their coefficients correspond to a one-standard-deviation change. One-hot encoded categorical coefficients are relative to the omitted/reference encoding created by the pipeline. These remain **predictive associations**, not causal effects.

## 11. Export compact Streamlit datasets

In [ ]:
exports = {
    "vintage_summary.csv": vintage_summary,
    "model_performance.csv": performance,
    "risk_deciles.csv": risk_deciles,
    "segment_summary.csv": segment_summary,
    "economic_groups.csv": economic_groups,
    "state_summary.csv": state_summary,
    "model_coefficients.csv": model_coefficients,
}

for filename, df in exports.items():
    path = APP_DATA_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved {filename:<28} {len(df):>6,} rows")

print("\nTotal exported size:")
total_bytes = sum((APP_DATA_DIR / name).stat().st_size for name in exports)
print(f"{total_bytes / 1024:.1f} KB")


## 12. Export app metadata and responsible-use text

In [ ]:
metadata = {
    "project_title": "Mortgage Risk Under Changing Economic Conditions",
    "dataset": "Freddie Mac Single-Family Loan-Level Dataset samples",
    "vintages": [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022],
    "mortgages": int(len(dashboard)),
    "target": "90+ day serious delinquency within the first 24 months",
    "training_vintages": "2015–2019",
    "validation_vintage": "2020",
    "test_vintages": "2021–2022",
    "model": "Logistic Regression",
    "model_feature_count": len(model_features),
    "responsible_use": (
        "This project is an educational portfolio analysis of historical mortgage "
        "performance. Results describe statistical associations and model behavior "
        "across historical loan cohorts and economic environments. They should not "
        "be interpreted as causal effects or used to make individual lending, "
        "underwriting, pricing, or eligibility decisions."
    ),
}

with open(APP_DATA_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 13. Final validation

In [ ]:
required_files = list(exports) + ["metadata.json"]

for filename in required_files:
    path = APP_DATA_DIR / filename
    assert path.exists(), f"Missing export: {filename}"
    assert path.stat().st_size > 0, f"Empty export: {filename}"

assert vintage_summary["origination_vintage"].tolist() == list(range(2015, 2023))
assert set(performance["vintage"]) == {2020, 2021, 2022}
assert risk_deciles["risk_decile"].between(1, 10).all()
assert dashboard["loan_id"].nunique() == len(dashboard)

print("All Streamlit data exports passed validation.")
print("\nReady for app.py.")


## Output

After running all cells, `data/app/` should contain:

```text
data/app/
├── vintage_summary.csv
├── model_performance.csv
├── risk_deciles.csv
├── segment_summary.csv
├── economic_groups.csv
├── state_summary.csv
├── model_coefficients.csv
└── metadata.json
```

These files are intentionally compact and aggregated so the deployed Streamlit app does not need the raw Freddie Mac files, the 397K-row modeling table, or individual loan identifiers.
